# 1. Prepare data

Downloads FER2013, does an 80/20 stratified train/val split, and writes
everything out as **physical folders** on Drive so the other two
notebooks can just point `ImageFolder` at them.

**Before running:** upload `fer_common.py` into your Drive project folder
(`DRIVE_ROOT` below) — it's the shared module the other notebooks import.

Run this notebook **once**. Re-run only if you want to redo the split.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os

DRIVE_ROOT = "/content/drive/MyDrive/comparison/boilerplate/"
os.makedirs(DRIVE_ROOT, exist_ok=True)
sys.path.append(DRIVE_ROOT)

import fer_common as fc

paths = fc.get_paths(DRIVE_ROOT)
for d in [paths["train_dir"], paths["val_dir"], paths["test_dir"], paths["weights_dir"]]:
    os.makedirs(d, exist_ok=True)


Mounted at /content/drive


In [2]:
!pip install -q kagglehub


## Download FER2013 (local, fast) then split onto Drive

In [3]:
import kagglehub

local_path = kagglehub.dataset_download("msambare/fer2013")
print("Downloaded to:", local_path)


Using Colab cache for faster access to the 'fer2013' dataset.
Downloaded to: /kaggle/input/fer2013


## Stratified 80/20 split

Reads the original `train` folder's file list and labels (no image
decoding needed for this part), computes a stratified split, then
**copies** each file into `data/train/<class>/` or `data/val/<class>/`
on Drive.

In [4]:
import shutil
import json
from collections import Counter

import numpy as np
import torch
from torchvision import datasets
from sklearn.model_selection import StratifiedShuffleSplit

local_train_dir = os.path.join(local_path, "train")
local_test_dir = os.path.join(local_path, "test")

dataset_for_split = datasets.ImageFolder(local_train_dir)
class_names = dataset_for_split.classes
targets = np.array(dataset_for_split.targets)
print("Classes:", class_names)
print("Total train+val images:", len(targets))

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(sss.split(np.zeros(len(targets)), targets))
print(f"Train: {len(train_idx)}  Val: {len(val_idx)}")


Classes: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
Total train+val images: 28709
Train: 22967  Val: 5742


In [5]:
def copy_split(indices, dest_dir, label):
    for i, idx in enumerate(indices):
        src_path, class_idx = dataset_for_split.samples[idx]
        class_name = class_names[class_idx]
        out_dir = os.path.join(dest_dir, class_name)
        os.makedirs(out_dir, exist_ok=True)
        shutil.copy(src_path, os.path.join(out_dir, os.path.basename(src_path)))
        if (i + 1) % 2000 == 0:
            print(f"  {label}: copied {i + 1}/{len(indices)}")
    print(f"{label}: done ({len(indices)} files)")

copy_split(train_idx, paths["train_dir"], "train")
copy_split(val_idx, paths["val_dir"], "val")


  train: copied 2000/22967
  train: copied 4000/22967
  train: copied 6000/22967
  train: copied 8000/22967
  train: copied 10000/22967
  train: copied 12000/22967
  train: copied 14000/22967
  train: copied 16000/22967
  train: copied 18000/22967
  train: copied 20000/22967
  train: copied 22000/22967
train: done (22967 files)
  val: copied 2000/5742
  val: copied 4000/5742
val: done (5742 files)


## Copy the test set as-is (already has per-class subfolders)

In [6]:
shutil.copytree(local_test_dir, paths["test_dir"], dirs_exist_ok=True)
print("Test set copied to", paths["test_dir"])


Test set copied to /content/drive/MyDrive/comparison/boilerplate/data/test


## Save class names + class weights for reuse by the other notebooks

In [7]:
train_targets = targets[train_idx]
class_counts = Counter(train_targets)
total_samples = sum(class_counts.values())

class_weights = torch.zeros(len(class_names), dtype=torch.float32)
for i in range(len(class_names)):
    if class_counts[i] > 0:
        class_weights[i] = total_samples / (len(class_names) * class_counts[i])

for i, name in enumerate(class_names):
    print(f"{name} weight:\t{class_weights[i]:.4f}")

with open(paths["class_names_path"], "w") as f:
    json.dump(class_names, f)

torch.save(class_weights, paths["class_weights_path"])

print("\nSaved class_names.json and class_weights.pt to", paths["data_dir"])


angry weight:	1.0266
disgust weight:	9.4011
fear weight:	1.0012
happy weight:	0.5684
neutral weight:	0.8260
sad weight:	0.8491
surprise weight:	1.2933

Saved class_names.json and class_weights.pt to /content/drive/MyDrive/comparison/boilerplate/data


In [18]:
!ls -la /content/drive/MyDrive/comparison/boilerplate/data/

total 15
-rw------- 1 root root   67 Aug 16 21:21 class_names.json
-rw------- 1 root root 1619 Aug 16 21:21 class_weights.pt
drwx------ 9 root root 4096 Jul  6 14:58 test
drwx------ 9 root root 4096 Aug 16 21:09 train
drwx------ 9 root root 4096 Aug 16 21:16 val


In [20]:
import os
test_dir = "/content/drive/MyDrive/comparison/boilerplate/data/test"
for cls in os.listdir(test_dir):
    n = len(os.listdir(os.path.join(test_dir, cls)))
    print(cls, n)

surprise 831
fear 1024
angry 958
neutral 1233
sad 1247
disgust 111
happy 1774


In [21]:
print(os.path.exists(paths["val_dir"]))
print(len(os.listdir(paths["val_dir"])) if os.path.exists(paths["val_dir"]) else "MISSING")

True
7


In [23]:
import shutil

LOCAL_TRAIN_DIR = "/content/data/train"
LOCAL_VAL_DIR = "/content/data/val"

if not os.path.exists(LOCAL_TRAIN_DIR):
    print("Copying train/val from Drive to local disk...")
    shutil.copytree(paths["train_dir"], LOCAL_TRAIN_DIR)
    shutil.copytree(paths["val_dir"], LOCAL_VAL_DIR)
    print("Done.")
else:
    print("Local data already present, skipping copy.")

Copying train/val from Drive to local disk...
Done.


In [24]:
LOCAL_TEST_DIR = "/content/data/test"

if not os.path.exists(LOCAL_TEST_DIR):
    print("Copying test set from Drive to local disk...")
    shutil.copytree(paths["test_dir"], LOCAL_TEST_DIR)
    print("Done.")
else:
    print("Local data already present, skipping copy.")

Copying test set from Drive to local disk...
Done.


In [25]:
import shutil

LOCAL_CLASS_NAMES = "/content/data/class_names.json"
LOCAL_CLASS_WEIGHTS = "/content/data/class_weights.pt"

if not os.path.exists(LOCAL_CLASS_NAMES):
    shutil.copy(paths["class_names_path"], LOCAL_CLASS_NAMES)
if not os.path.exists(LOCAL_CLASS_WEIGHTS):
    shutil.copy(paths["class_weights_path"], LOCAL_CLASS_WEIGHTS)

with open(LOCAL_CLASS_NAMES) as f:
    class_names = json.load(f)
class_weights = torch.load(LOCAL_CLASS_WEIGHTS)

## Done

`data/train`, `data/val`, `data/test` on Drive now hold the split images,
and `class_names.json` / `class_weights.pt` are saved alongside them.
Notebook 2 can now train any backbone against this data — you don't need
to run this notebook again unless you want to change the split.